In [2]:
!pip install joblib numpy pandas matplotlib xgboost scikit-learn

  Using cached numpy-2.5.3-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pillow-12.3.0-cp313-cp313-win_amd64.whl.metadata (9.3 kB)
Using cached numpy-2.5.3-cp313-cp313-win_amd64.whl (12.6 MB)
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ------------------ --------------------- 4.5/9.6 MB 21.1 MB/s eta 0:00:01
   ------------------------------ --------- 7.3/9.6 MB 25.6 MB/s eta 0:00:01
   ------------------------------ --------- 7.3/9.6 MB 25.6 MB/s eta 0:00:01
   --------------------------------- ------ 8.1/9.6 MB 10.1 MB/s eta 0:00:01
   ----------------------------------- ---- 8.7/9.6 MB 8.8 MB/s eta 0:00:01
   -------------------------------------- - 9.2/9.6 MB 7.7 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.6 MB 7.0 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 5.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from src.config import (
    RAW_DATA_PATH,
    FEATURED_DATA_PATH,
    MODEL_PATHS,
    PREDS_PATHS,
    FEATURE_COLS,
    TARGET_COL,
    DATE_COL,
)

ModuleNotFoundError: No module named 'src'

In [ ]:
df = pd.read_csv(
    FEATURED_DATA_PATH,
    parse_dates=[DATE_COL]
)

df = df.sort_values(
    ["store", "item", DATE_COL]
).reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Date range:", df[DATE_COL].min(), "to", df[DATE_COL].max())

In [ ]:
train_df = df[df[DATE_COL] < "2017-01-01"].copy()
val_df = df[df[DATE_COL] >= "2017-01-01"].copy()

print("Training rows:", len(train_df))
print("Validation rows:", len(val_df))

In [ ]:
X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET_COL]

X_val = val_df[FEATURE_COLS]
y_val = val_df[TARGET_COL]

print("Features:")
print(FEATURE_COLS)

print("\nTraining shape:", X_train.shape)
print("Validation shape:", X_val.shape)

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train
)

print("XGBoost training completed.")

In [ ]:
val_preds = xgb_model.predict(X_val)

val_preds = np.clip(
    val_preds,
    0,
    None
)

print("Predictions generated:", len(val_preds))

In [ ]:
mae = mean_absolute_error(
    y_val,
    val_preds
)

rmse = np.sqrt(
    mean_squared_error(
        y_val,
        val_preds
    )
)

non_zero = y_val != 0

mape = (
    np.mean(
        np.abs(
            (y_val[non_zero] - val_preds[non_zero])
            / y_val[non_zero]
        )
    ) * 100
)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAPE : {mape:.4f}%")

In [ ]:
importance = (
    pd.Series(
        xgb_model.feature_importances_,
        index=FEATURE_COLS
    )
    .sort_values(ascending=True)
)

plt.figure(figsize=(8, 6))

importance.plot(
    kind="barh"
)

plt.title("XGBoost Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()

os.makedirs(
    os.path.join(
        os.path.dirname(os.path.dirname(FEATURED_DATA_PATH)),
        "plots",
        "xgboost_plots"
    ),
    exist_ok=True
)

plt.savefig(
    os.path.join(
        os.path.dirname(os.path.dirname(FEATURED_DATA_PATH)),
        "plots",
        "xgboost_plots",
        "xgboost_feature_importance.png"
    ),
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(12, 4))

plt.plot(
    y_val.values[:200],
    label="Actual"
)

plt.plot(
    val_preds[:200],
    label="Predicted"
)

plt.xlabel("Time Steps")
plt.ylabel("Sales")
plt.title("XGBoost Forecast vs Actual")

plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(
        os.path.dirname(os.path.dirname(FEATURED_DATA_PATH)),
        "plots",
        "xgboost_plots",
        "xgboost_actual_vs_predicted.png"
    ),
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
validation_results = val_df[
    [DATE_COL, "store", "item", TARGET_COL]
].copy()

validation_results["prediction"] = val_preds

validation_results["error"] = (
    validation_results[TARGET_COL]
    - validation_results["prediction"]
)

validation_results["absolute_error"] = (
    validation_results["error"]
    .abs()
)

validation_results.head()

In [ ]:
os.makedirs(
    os.path.dirname(MODEL_PATHS["XGBoost"]),
    exist_ok=True
)

joblib.dump(
    xgb_model,
    MODEL_PATHS["XGBoost"]
)

joblib.dump(
    val_preds,
    PREDS_PATHS["XGBoost"]
)

joblib.dump(
    y_val.to_numpy(),
    PREDS_PATHS["actual"]
)

# Portable XGBoost format
json_path = os.path.join(
    os.path.dirname(MODEL_PATHS["XGBoost"]),
    "xgb_model.json"
)

xgb_model.save_model(
    json_path
)

print("Saved:")
print("-", MODEL_PATHS["XGBoost"])
print("-", PREDS_PATHS["XGBoost"])
print("-", PREDS_PATHS["actual"])
print("-", json_path)

In [ ]:
print("=" * 50)
print("INVENTI XGBOOST MODEL SUMMARY")
print("=" * 50)

print(f"Training rows   : {len(train_df):,}")
print(f"Validation rows : {len(val_df):,}")
print(f"Features        : {len(FEATURE_COLS)}")
print(f"MAE             : {mae:.4f}")
print(f"RMSE            : {rmse:.4f}")
print(f"MAPE            : {mape:.4f}%")
print("=" * 50)